In [1]:
import os
from dotenv import load_dotenv
from qdrant_client import QdrantClient
from sentence_transformers import SentenceTransformer
from openai import OpenAI
import textwrap
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage


load_dotenv()


e:\Crail 2025\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
GROQ_MODEL = os.getenv("GROQ_MODEL")
QDRANT_PORT = 6334 
COLLECTION_NAME = "Crail_data"

In [3]:
llm = ChatGroq(
    model=GROQ_MODEL,
    api_key=GROQ_API_KEY,
    temperature=0.5,
    max_tokens=5000,
    top_p=0.95,
    frequency_penalty=0,
    presence_penalty=0,
    stop=None,
)

e:\Crail 2025\.venv\Lib\site-packages\langchain_groq\chat_models.py:370: UserWarning: WARNING! top_p is not default parameter.
                    top_p was transferred to model_kwargs.
                    Please confirm that top_p is what you intended.
  warnings.warn(
e:\Crail 2025\.venv\Lib\site-packages\langchain_groq\chat_models.py:370: UserWarning: WARNING! frequency_penalty is not default parameter.
                    frequency_penalty was transferred to model_kwargs.
                    Please confirm that frequency_penalty is what you intended.
  warnings.warn(
e:\Crail 2025\.venv\Lib\site-packages\langchain_groq\chat_models.py:370: UserWarning: WARNING! presence_penalty is not default parameter.
                    presence_penalty was transferred to model_kwargs.
                    Please confirm that presence_penalty is what you intended.
  warnings.warn(


In [4]:
qdrant = QdrantClient("http://localhost", port=QDRANT_PORT)  # Use your secondary container port
embedding_model  = SentenceTransformer("all-MiniLM-L6-v2")

In [5]:
def search_qdrant(user_query, top_k=5):
    query_vector = embedding_model.encode(user_query).tolist()
    results = qdrant.search(
        collection_name=COLLECTION_NAME,
        query_vector=query_vector,
        limit=top_k
    )
    return results

In [6]:
def ask_llm(question, context):
    prompt = f"""
You are a helpful hospital assistant.

Use the following context to answer the user's question.

Context:
{textwrap.indent(context, '  ')}

Question:
{question}

Answer:"""

    response = llm.invoke([HumanMessage(content=prompt)])
    return response.content.strip()

In [7]:
def query_system(user_query):
    print(f"\n🔍 User Question: {user_query}\n")

    results = search_qdrant(user_query)
    context_chunks = []

    for res in results:
        payload = res.payload.get("original_data")
        if payload:
            context_chunks.append(str(payload))

    if not context_chunks:
        return "No relevant documents found in the database."

    combined_context = "\n\n".join(context_chunks)
    answer = ask_llm(user_query, combined_context)

    print("\n💬 LLM Response:")
    print(textwrap.fill(answer, width=100))

In [8]:
"Who is the most experienced cardiologist?"

"What treatments are available for heart-related issues?"

"Can I consult Dr. Rajiv Kumar online?"

"How much does knee replacement cost?"

'How much does knee replacement cost?'

In [10]:
question = "Dr. Anjali Mehta's consultation fee and available timings on Fridays."
query_system(question)


🔍 User Question: Dr. Anjali Mehta's consultation fee and available timings on Fridays.



C:\Users\Rajeev Bandi\AppData\Local\Temp\ipykernel_27068\3655553256.py:3: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  results = qdrant.search(


'No relevant documents found in the database.'